# WMDP Classifier-Gated PoRT Mini GPU Test

This notebook validates the canonical `llm-unlearn-eco/scripts/evaluate_wmdp.py` entrypoint in classifier-gated mode from a Kaggle-style clean clone. It requires a local Hugging Face text-classification model directory mounted on Kaggle and exposed through `WMDP_CLASSIFIER_PATH` or `MANUAL_CLASSIFIER_PATH`.

The run uses `multiple_choice_zero_out.yaml`, `sample_size=2`, and intentionally does not pass `--attack_all_prompts`.

In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()

def has_project_layout(path):
    path = Path(path)
    return (path / 'llm-unlearn-eco' / 'eco').exists() and (path / 'dataset').exists()

def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()

PROJECT_ROOT = clone_or_use_project()
ECO_ROOT = PROJECT_ROOT / 'llm-unlearn-eco'
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
if str(ECO_ROOT) not in sys.path:
    sys.path.insert(0, str(ECO_ROOT))

commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')

In [ ]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'safetensors': 'safetensors',
    'tabulate': 'tabulate',
    'tqdm': 'tqdm',
    'transformers': 'transformers>=4.38.0',
    'accelerate': 'accelerate>=0.26.0',
    'huggingface_hub': 'huggingface_hub',
    'yaml': 'pyyaml',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')

In [ ]:
import platform
import torch

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('This classifier-gated smoke test is intended for a Kaggle GPU runtime.')
for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    print(f'GPU {idx}: {props.name}, VRAM={props.total_memory / 1024**3:.2f} GiB')

## Resolve Or Build Classifier Artifact

Set `MANUAL_CLASSIFIER_PATH` if a local classifier directory already exists. Otherwise the notebook can materialize a real classifier into `/kaggle/working` without Kaggle Input:

- `PORT_WMDP_CLASSIFIER_HF_REPO`: download a Hugging Face model repo with `snapshot_download`.
- `PORT_WMDP_CLASSIFIER_ARCHIVE_URL`: download and extract a `.zip`, `.tar`, `.tar.gz`, or `.tgz` archive.
- Default: train a lightweight WMDP-vs-non-WMDP text classifier from repo-local WMDP positives and TOFU negatives, then save it as a local Hugging Face model directory.

The default auto-trained classifier is a real trained classifier for pipeline validation. For final experiment numbers, replace it with the intended classifier artifact or a locked classifier download URL/repo.

In [ ]:
MANUAL_CLASSIFIER_PATH = ''  # Example: '/kaggle/working/wmdp_classifier_model'
CLASSIFIER_MODEL_NAME = os.environ.get('PORT_WMDP_CLASSIFIER_MODEL_NAME', 'wmdp_classifier_auto_trained')
CLASSIFIER_THRESHOLD = float(os.environ.get('PORT_WMDP_CLASSIFIER_THRESHOLD', '0.5'))
CLASSIFIER_BATCH_SIZE = int(os.environ.get('PORT_CLASSIFIER_BATCH_SIZE', '1'))

CLASSIFIER_HF_REPO = os.environ.get('PORT_WMDP_CLASSIFIER_HF_REPO', '').strip()
CLASSIFIER_HF_REVISION = os.environ.get('PORT_WMDP_CLASSIFIER_HF_REVISION', '').strip() or None
CLASSIFIER_ARCHIVE_URL = os.environ.get('PORT_WMDP_CLASSIFIER_ARCHIVE_URL', '').strip()
CLASSIFIER_BACKBONE = os.environ.get('PORT_WMDP_CLASSIFIER_BACKBONE', 'prajjwal1/bert-tiny')
AUTO_TRAIN_CLASSIFIER = os.environ.get('PORT_AUTO_TRAIN_WMDP_CLASSIFIER', '1').strip().lower() not in {'0', 'false', 'no'}

MATERIALIZED_CLASSIFIER_DIR = Path(
    os.environ.get(
        'PORT_MATERIALIZED_CLASSIFIER_DIR',
        '/kaggle/working/wmdp_prompt_classifier' if IS_KAGGLE else str(PROJECT_ROOT / 'results' / 'wmdp_prompt_classifier'),
    )
)

WEIGHT_FILENAMES = {
    'model.safetensors',
    'model.safetensors.index.json',
    'pytorch_model.bin',
    'pytorch_model.bin.index.json',
}
TOKENIZER_FILENAMES = {
    'tokenizer.json',
    'tokenizer_config.json',
    'vocab.json',
    'vocab.txt',
    'spiece.model',
}
INTERESTING_FILENAMES = WEIGHT_FILENAMES | TOKENIZER_FILENAMES | {
    'config.json',
    'adapter_model.safetensors',
    'training_args.bin',
}

def looks_like_hf_model_dir(path):
    path = Path(path)
    if not (path / 'config.json').exists():
        return False
    has_tokenizer = any((path / name).exists() for name in TOKENIZER_FILENAMES)
    has_weights = any((path / name).exists() for name in WEIGHT_FILENAMES)
    has_sharded_weights = bool(list(path.glob('model-*-of-*.safetensors'))) or bool(list(path.glob('pytorch_model-*-of-*.bin')))
    return has_tokenizer and (has_weights or has_sharded_weights)

def find_classifier_candidates(root):
    root = Path(root)
    if not root.exists():
        return []
    candidates = []
    for config_path in root.rglob('config.json'):
        candidate = config_path.parent
        if looks_like_hf_model_dir(candidate):
            score_text = str(candidate).lower()
            score = int('wmdp' in score_text) + int('classifier' in score_text)
            candidates.append((score, candidate))
    candidates.sort(key=lambda item: (-item[0], len(str(item[1]))))
    return [candidate for _, candidate in candidates]

def summarize_kaggle_inputs(root='/kaggle/input', max_files=80):
    root = Path(root)
    if not root.exists():
        return 'No /kaggle/input directory is available.'
    lines = ['Mounted Kaggle inputs:']
    children = sorted(root.iterdir())
    if not children:
        lines.append('  /kaggle/input is empty.')
        return '\n'.join(lines)
    for child in children:
        lines.append(f'  {child}')
    interesting = []
    for file_path in root.rglob('*'):
        if file_path.is_file() and (
            file_path.name in INTERESTING_FILENAMES
            or file_path.name.endswith('.safetensors')
            or file_path.name.endswith('.bin')
        ):
            interesting.append(file_path)
            if len(interesting) >= max_files:
                break
    if interesting:
        lines.append('Interesting model-related files found:')
        for file_path in interesting:
            lines.append(f'  {file_path}')
    else:
        lines.append('No config/tokenizer/weight files were found under /kaggle/input.')
    return '\n'.join(lines)

def materialize_classifier_from_hf(repo_id, target_dir, revision=None):
    from huggingface_hub import snapshot_download

    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=repo_id,
        revision=revision,
        local_dir=str(target_dir),
        local_dir_use_symlinks=False,
    )
    return target_dir

def materialize_classifier_from_archive(url, target_dir):
    import shutil
    import tarfile
    import urllib.request
    import zipfile

    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    archive_path = target_dir / Path(url.split('?')[0]).name
    urllib.request.urlretrieve(url, archive_path)

    extract_dir = target_dir / 'extracted'
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    name = archive_path.name.lower()
    if name.endswith('.zip'):
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(extract_dir)
    elif name.endswith(('.tar.gz', '.tgz', '.tar')):
        with tarfile.open(archive_path) as tf:
            tf.extractall(extract_dir)
    else:
        raise RuntimeError(f'Unsupported classifier archive type: {archive_path.name}')

    candidates = find_classifier_candidates(extract_dir)
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one classifier directory inside archive, found {len(candidates)}: {candidates[:10]}')
    return candidates[0]

def load_json_rows(path, limit=None):
    rows = json.loads(Path(path).read_text(encoding='utf-8'))
    return rows[:limit] if limit else rows

def format_negative_prompt(row):
    question = str(row.get('question', '')).strip()
    options = [row.get(f'option{i}') for i in range(1, 5)]
    if all(option is not None for option in options):
        choice_lines = '\n'.join(f'{label}. {option}' for label, option in zip(['A', 'B', 'C', 'D'], options))
        return f'The following are multiple choice questions (with answers) about general knowledge.\n\nQuestion:\n{question}\n{choice_lines}\n\nAnswer:'
    return f'Question:\n{question}\n\nAnswer:'

def build_classifier_training_texts(max_pos_per_domain=384, max_neg=1152):
    from eco.dataset import WMDPBio, WMDPChem, WMDPCyber

    positives = []
    for dataset_cls in [WMDPBio, WMDPChem, WMDPCyber]:
        dataset = dataset_cls(sample_size=max_pos_per_domain).load_dataset_for_eval('test')
        positives.extend(str(row['prompt']) for row in dataset)

    negative_files = [
        PROJECT_ROOT / 'dataset' / 'TOFU' / 'original' / 'world_facts.json',
        PROJECT_ROOT / 'dataset' / 'TOFU' / 'original' / 'retain90.json',
        PROJECT_ROOT / 'dataset' / 'TOFU' / 'original' / 'real_authors.json',
    ]
    negatives = []
    for file_path in negative_files:
        if file_path.exists():
            for row in load_json_rows(file_path):
                negatives.append(format_negative_prompt(row))
                if len(negatives) >= max_neg:
                    break
        if len(negatives) >= max_neg:
            break

    target_size = min(len(positives), len(negatives))
    if target_size < 32:
        raise RuntimeError(f'Not enough classifier training data: positives={len(positives)}, negatives={len(negatives)}')
    return positives[:target_size], negatives[:target_size]

def train_wmdp_prompt_classifier(output_dir):
    import random
    import time
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    output_dir = Path(output_dir)
    if looks_like_hf_model_dir(output_dir):
        print(f'Using existing auto-trained classifier: {output_dir}')
        return output_dir

    max_pos_per_domain = int(os.environ.get('PORT_WMDP_CLASSIFIER_POS_PER_DOMAIN', '384'))
    max_neg = int(os.environ.get('PORT_WMDP_CLASSIFIER_MAX_NEG', str(max_pos_per_domain * 3)))
    epochs = int(os.environ.get('PORT_WMDP_CLASSIFIER_EPOCHS', '2'))
    train_batch_size = int(os.environ.get('PORT_WMDP_CLASSIFIER_TRAIN_BATCH_SIZE', '16'))
    max_length = int(os.environ.get('PORT_WMDP_CLASSIFIER_MAX_LENGTH', '256'))
    learning_rate = float(os.environ.get('PORT_WMDP_CLASSIFIER_LR', '2e-5'))

    positives, negatives = build_classifier_training_texts(max_pos_per_domain=max_pos_per_domain, max_neg=max_neg)
    examples = [(text, 1) for text in positives] + [(text, 0) for text in negatives]
    rng = random.Random(0)
    rng.shuffle(examples)
    split_idx = max(1, int(len(examples) * 0.9))
    train_examples = examples[:split_idx]
    valid_examples = examples[split_idx:]

    tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_BACKBONE)
    model = AutoModelForSequenceClassification.from_pretrained(
        CLASSIFIER_BACKBONE,
        num_labels=2,
        id2label={0: 'LABEL_0', 1: 'LABEL_1'},
        label2id={'LABEL_0': 0, 'LABEL_1': 1},
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token
        model.config.pad_token_id = tokenizer.pad_token_id

    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    def encode_dataset(rows):
        texts = [text for text, _ in rows]
        labels = torch.tensor([label for _, label in rows], dtype=torch.long)
        encoded = tokenizer(texts, truncation=True, padding=True, max_length=max_length, return_tensors='pt')
        return TensorDataset(encoded['input_ids'], encoded['attention_mask'], labels)

    train_loader = DataLoader(encode_dataset(train_examples), batch_size=train_batch_size, shuffle=True)
    valid_loader = DataLoader(encode_dataset(valid_examples), batch_size=train_batch_size)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    started = time.perf_counter()
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for input_ids, attention_mask, labels in train_loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            outputs.loss.backward()
            optimizer.step()
            total_loss += float(outputs.loss.detach().cpu())

        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for input_ids, attention_mask, labels in valid_loader:
                input_ids = input_ids.to(device)
                attention_mask = attention_mask.to(device)
                labels = labels.to(device)
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                preds = logits.argmax(dim=-1)
                correct += int((preds == labels).sum().item())
                total += int(labels.numel())
        val_acc = correct / total if total else 0.0
        print(f'Classifier epoch {epoch + 1}/{epochs}: loss={total_loss / max(len(train_loader), 1):.4f}, val_acc={val_acc:.4f}')

    output_dir.mkdir(parents=True, exist_ok=True)
    tokenizer.save_pretrained(str(output_dir))
    model.save_pretrained(str(output_dir), safe_serialization=True)
    summary = {
        'source': 'auto_trained_from_repo_data',
        'backbone': CLASSIFIER_BACKBONE,
        'num_positive': len(positives),
        'num_negative': len(negatives),
        'num_train': len(train_examples),
        'num_valid': len(valid_examples),
        'epochs': epochs,
        'threshold': CLASSIFIER_THRESHOLD,
        'seconds': time.perf_counter() - started,
    }
    (output_dir / 'classifier_training_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(f'Wrote trained classifier to: {output_dir}')
    print(json.dumps(summary, indent=2))
    return output_dir

classifier_path_value = (
    MANUAL_CLASSIFIER_PATH.strip()
    or os.environ.get('WMDP_CLASSIFIER_PATH', '').strip()
    or os.environ.get('PORT_WMDP_CLASSIFIER_PATH', '').strip()
)
CLASSIFIER_SOURCE = 'manual_or_env'

if not classifier_path_value and IS_KAGGLE:
    candidates = find_classifier_candidates('/kaggle/input')
    if len(candidates) == 1:
        classifier_path_value = str(candidates[0])
        CLASSIFIER_SOURCE = 'kaggle_input_auto_detected'
        print(f'Auto-detected classifier path: {classifier_path_value}')
    elif candidates:
        print('Multiple classifier-like model directories found:')
        for candidate in candidates[:20]:
            print('  ', candidate)
        raise RuntimeError('Set MANUAL_CLASSIFIER_PATH to the intended WMDP classifier directory.')

if not classifier_path_value and CLASSIFIER_HF_REPO:
    classifier_path_value = str(materialize_classifier_from_hf(CLASSIFIER_HF_REPO, MATERIALIZED_CLASSIFIER_DIR, revision=CLASSIFIER_HF_REVISION))
    CLASSIFIER_SOURCE = f'hf_repo:{CLASSIFIER_HF_REPO}'

if not classifier_path_value and CLASSIFIER_ARCHIVE_URL:
    classifier_path_value = str(materialize_classifier_from_archive(CLASSIFIER_ARCHIVE_URL, MATERIALIZED_CLASSIFIER_DIR))
    CLASSIFIER_SOURCE = f'archive_url:{CLASSIFIER_ARCHIVE_URL}'

if not classifier_path_value:
    print(summarize_kaggle_inputs())
    if not AUTO_TRAIN_CLASSIFIER:
        raise RuntimeError(
            'No WMDP classifier directory was configured or auto-detected. '
            'Set PORT_WMDP_CLASSIFIER_HF_REPO, PORT_WMDP_CLASSIFIER_ARCHIVE_URL, '
            'WMDP_CLASSIFIER_PATH, MANUAL_CLASSIFIER_PATH, or enable PORT_AUTO_TRAIN_WMDP_CLASSIFIER=1.'
        )
    classifier_path_value = str(train_wmdp_prompt_classifier(MATERIALIZED_CLASSIFIER_DIR))
    CLASSIFIER_SOURCE = 'auto_trained_from_repo_data'

CLASSIFIER_PATH = Path(classifier_path_value).expanduser().resolve()
if not CLASSIFIER_PATH.exists():
    print(summarize_kaggle_inputs())
    raise FileNotFoundError(f'Classifier path does not exist: {CLASSIFIER_PATH}')
if not CLASSIFIER_PATH.is_dir():
    raise NotADirectoryError(f'Classifier path must be a directory: {CLASSIFIER_PATH}')
if not looks_like_hf_model_dir(CLASSIFIER_PATH):
    print('Classifier directory contents:')
    for child in sorted(CLASSIFIER_PATH.iterdir()):
        print('  ', child.name)
    raise RuntimeError(
        'Classifier path does not look like a local Hugging Face model directory. '
        'Use a directory containing config.json, tokenizer files, and model.safetensors or pytorch_model.bin.'
    )

os.environ['WMDP_CLASSIFIER_PATH'] = str(CLASSIFIER_PATH)
print(f'WMDP_CLASSIFIER_PATH={CLASSIFIER_PATH}')
print(f'Classifier source: {CLASSIFIER_SOURCE}')
print(f'Classifier model name: {CLASSIFIER_MODEL_NAME}')
print(f'Classifier threshold: {CLASSIFIER_THRESHOLD}')

## Load Classifier Once

In [ ]:
from eco.attack import PromptClassifier
from eco.dataset import WMDPBio
from eco.utils import delete_model

bio = WMDPBio(sample_size=1)
wmdp_prompt = bio.load_dataset_for_eval('test')[0]['prompt']
probe_prompts = [
    wmdp_prompt,
    'What is the capital of France? Answer:',
]

classifier = PromptClassifier(
    model_name=CLASSIFIER_MODEL_NAME,
    model_path=str(CLASSIFIER_PATH),
    batch_size=CLASSIFIER_BATCH_SIZE,
)
try:
    labels = classifier.predict(probe_prompts, threshold=CLASSIFIER_THRESHOLD)
    print('Classifier probe labels:', labels)
    if all(label == 0 for label in labels) or all(label == 1 for label in labels):
        print('WARNING: probe labels are all identical; inspect threshold and label mapping before full runs.')
finally:
    delete_model(classifier)

## Run Canonical Script

In [ ]:
TARGET_CONFIG_NAME = os.environ.get('PORT_TARGET_CONFIG_NAME', 'phi-1_5')
TARGET_HF_NAME = os.environ.get('PORT_TARGET_HF_NAME')
TARGET_MODEL_PATH = os.environ.get('PORT_TARGET_MODEL_PATH')
ATTN_IMPLEMENTATION = os.environ.get('PORT_ATTN_IMPLEMENTATION', 'eager')
TORCH_DTYPE = os.environ.get('PORT_TORCH_DTYPE', 'float16')
BATCH_SIZE = int(os.environ.get('PORT_WMDP_BATCH_SIZE', '1'))
SAMPLE_SIZE = int(os.environ.get('PORT_WMDP_SAMPLE_SIZE', '2'))
TASK_CONFIG = os.environ.get('PORT_WMDP_TASK_CONFIG', 'multiple_choice_zero_out.yaml')
RUN_NAME = os.environ.get(
    'PORT_RUN_NAME',
    f'wmdp_script_mini_zero_out_classifier_gated_{TARGET_CONFIG_NAME}',
)

if IS_KAGGLE:
    print(f'Refreshing repository at {PROJECT_ROOT}')
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'])

SCRIPT_PATH = PROJECT_ROOT / 'llm-unlearn-eco' / 'scripts' / 'evaluate_wmdp.py'
OUTPUT_DIR = Path('/kaggle/working') if IS_KAGGLE else PROJECT_ROOT / 'results'
script_source = SCRIPT_PATH.read_text(encoding='utf-8')
required_script_markers = ['--attack_all_prompts', 'resolve_classifier_path', 'WMDP_CLASSIFIER_PATH']
missing_script_markers = [marker for marker in required_script_markers if marker not in script_source]
if missing_script_markers:
    raise RuntimeError(
        'The cloned repository does not contain the classifier-gated evaluate_wmdp.py updates yet. '
        f'Missing markers: {missing_script_markers}. Pull or push the latest repo commit.'
    )

SCRIPT_ENV = os.environ.copy()
existing_pythonpath = SCRIPT_ENV.get('PYTHONPATH')
SCRIPT_ENV['PYTHONPATH'] = str(ECO_ROOT) if not existing_pythonpath else str(ECO_ROOT) + os.pathsep + existing_pythonpath
SCRIPT_ENV['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
SCRIPT_ENV['WMDP_CLASSIFIER_PATH'] = str(CLASSIFIER_PATH)

cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    '--model_name', TARGET_CONFIG_NAME,
    '--task_config', TASK_CONFIG,
    '--sample_size', str(SAMPLE_SIZE),
    '--batch_size', str(BATCH_SIZE),
    '--torch_dtype', TORCH_DTYPE,
    '--attn_implementation', ATTN_IMPLEMENTATION,
    '--classifier_path', str(CLASSIFIER_PATH),
    '--classifier_model_name', CLASSIFIER_MODEL_NAME,
    '--classifier_threshold', str(CLASSIFIER_THRESHOLD),
    '--output_dir', str(OUTPUT_DIR),
    '--run_name', RUN_NAME,
]
if TARGET_HF_NAME:
    cmd.extend(['--target_hf_name', TARGET_HF_NAME])
if TARGET_MODEL_PATH:
    cmd.extend(['--model_path', TARGET_MODEL_PATH])

print('$ ' + ' '.join(cmd))
subprocess.check_call(cmd, cwd=str(PROJECT_ROOT), env=SCRIPT_ENV)

## Verify Artifacts

In [ ]:
import pandas as pd

run_dir = OUTPUT_DIR / RUN_NAME
summary_path = run_dir / 'summary.json'
predictions_path = run_dir / 'predictions.csv'
attack_stats_path = run_dir / 'attack_stats.csv'
summary_by_run_path = run_dir / 'summary_by_run.csv'

for path in [summary_path, predictions_path, attack_stats_path, summary_by_run_path]:
    if not path.exists():
        raise FileNotFoundError(path)

summary = json.loads(summary_path.read_text(encoding='utf-8'))
predictions = pd.read_csv(predictions_path)
attack_stats = pd.read_csv(attack_stats_path)
summary_by_run = pd.read_csv(summary_by_run_path)

expected_rows = SAMPLE_SIZE * 3
if len(predictions) != expected_rows:
    raise AssertionError(f'Expected {expected_rows} prediction rows, got {len(predictions)}')

required_attack_columns = {'run_label', 'dataset', 'classifier_mode', 'num_prompts', 'num_attacked', 'attack_rate'}
missing_columns = required_attack_columns - set(attack_stats.columns)
if missing_columns:
    raise AssertionError(f'Missing attack_stats columns: {sorted(missing_columns)}')
if set(attack_stats['classifier_mode']) != {'classifier_gated'}:
    raise AssertionError(f'Unexpected classifier modes: {sorted(set(attack_stats["classifier_mode"]))}')

print('Prediction rows:', len(predictions))
print('Attack stats:')
print(attack_stats.to_string(index=False))
print('Summary by run:')
print(summary_by_run.to_string(index=False))

rates = attack_stats['attack_rate']
print('Classifier source:', CLASSIFIER_SOURCE)
print('Classifier path:', CLASSIFIER_PATH)
if (rates == 0).all():
    print('WARNING: attack_rate is all 0; lower threshold or inspect classifier labels before full WMDP.')
elif (rates == 1).all():
    print('NOTE: attack_rate is all 1 on this WMDP-only mini set; verify non-WMDP probe labels before utility/full runs.')

print('CLASSIFIER-GATED MINI TEST COMPLETED')